# Language Imbalance BabyLM — Experiments

Workflow:
1. Train one shared tokenizer on all three languages
2. Monolingual scaling law: train eng / dut / ind at increasing `--max_tokens`
3. Bilingual experiments: eng+dut at 50/50, 70/30, 90/10 with fixed token budget
4. Parse perplexity from training logs and plot

In [ ]:
import subprocess, re, json
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

## Config

In [ ]:
DATASETS = {
    "eng": "BabyLM-community/babylm-eng",
    "dut": "BabyLM-community/babylm-nld",
    "ind": "BabyLM-community/babylm-ind",
}

# Maps dataset name suffix (eval_lang key from train.py) → short lang key used here
EVAL_TO_LANG = {
    "babylm-eng": "eng",
    "babylm-nld": "dut",
    "babylm-ind": "ind",
}

CONFIG        = "./small_config.json"   # official BabyLM small model (~28M params)
TOKENIZER_DIR = "./shared_tokenizer"
MAX_LENGTH    = 128
SEED          = 42
BATCH_SIZE    = 32
EPOCHS        = 3

# Token budgets for scaling law runs (same for all languages → fair comparison)
SCALING_BUDGETS = [100_000, 500_000, 1_000_000, 5_000_000, 10_000_000]

# Bilingual token budget (fixed — same total tokens as largest monolingual run)
BILINGUAL_BUDGET = 10_000_000
BILINGUAL_RATIOS = [
    ("eng", "dut", [0.5, 0.5]),
    ("eng", "dut", [0.7, 0.3]),
    ("eng", "dut", [0.9, 0.1]),
]

## Helper

In [ ]:
def run(cmd: str):
    """Run a shell command, stream output to stdout."""
    print(f">>> {cmd}\n")
    return subprocess.run(cmd, shell=True, executable="/bin/bash", capture_output=False, text=True)


def parse_perplexity(stdout: str) -> dict[str, float]:
    """Extract per-language perplexity from train.py stdout.

    train.py prints lines like:  babylm-eng: perplexity = 42.31
    Returns short lang keys, e.g. {"eng": 42.31}.
    """
    pattern = r"(\S+):\s*perplexity\s*=\s*([\d.]+)"
    return {
        EVAL_TO_LANG.get(m.group(1), m.group(1)): float(m.group(2))
        for m in re.finditer(pattern, stdout)
    }


def train_cmd(
    langs: list[str],
    output_dir: str,
    model_name: str,
    max_tokens: int | None = None,
    ratios: list[float] | None = None,
    epochs: int = 3,
) -> str:
    dataset_str = " ".join(DATASETS[lang] for lang in langs)
    cmd = (
        f"python train.py"
        f" --dataset {dataset_str}"
        f" --config {CONFIG}"
        f" --tokenizer_dir {TOKENIZER_DIR}"
        f" --output_dir {output_dir}"
        f" --model_name {model_name}"
        f" --max_length {MAX_LENGTH}"
        f" --batch_size {BATCH_SIZE}"
        f" --epochs {epochs}"
        f" --seed {SEED}"
    )
    if max_tokens is not None:
        cmd += f" --max_tokens {max_tokens}"
    if ratios is not None:
        cmd += " --ratios " + " ".join(str(r) for r in ratios)
    return cmd

## 1 — Shared tokenizer (run once)

Trains a BPE tokenizer on all three languages combined.
All subsequent runs load it via `--tokenizer_dir`, so perplexity numbers are directly comparable.

In [14]:
dataset_str = " ".join(DATASETS.values())
cmd = (
    f"python train.py"
    f" --dataset {dataset_str}"
    f" --config {CONFIG}"
    f" --output_dir {TOKENIZER_DIR}"
    f" --model_name unused"
    f" --max_length {MAX_LENGTH}"
    f" --seed {SEED}"
    f" --epochs 0"  # exits after saving tokenizer
)
run(cmd)

>>> python train.py --dataset BabyLM-community/babylm-eng BabyLM-community/babylm-nld BabyLM-community/babylm-ind --config ./small_config.json --output_dir ./shared_tokenizer --model_name unused --max_length 128 --seed 42 --epochs 0

📥 Loading datasets...
  babylm-eng: 137710 rows
  babylm-nld: 304611 rows
  babylm-ind: 37704 rows
  babylm-eng: 5000 eval rows, 132710 pool rows
  babylm-nld: 5000 eval rows, 299611 pool rows
  babylm-ind: 1885 eval rows, 35819 pool rows
Source documents available: 468140
🔡 Training SentencePiece tokenizer...


KeyboardInterrupt: 

/Users/b.leucht/.local/share/uv/python/cpython-3.11.15-macos-aarch64-none/lib/python3.11/multiprocessing/resource_tracker.py:254: UserWarning: resource_tracker: There appear to be 1 leaked semaphore objects to clean up at shutdown
  warnings.warn('resource_tracker: There appear to be %d '


## 2 — Monolingual scaling law runs

Each language is trained at every token budget in `SCALING_BUDGETS`.
Results are stored in `scaling_results` for plotting.

In [ ]:
scaling_results = []  # list of dicts: {lang, max_tokens, perplexity}

for lang in DATASETS:
    for budget in SCALING_BUDGETS:
        output_dir = f"./mono-{lang}-{budget // 1_000}k"
        cmd = train_cmd(
            langs=[lang],
            output_dir=output_dir,
            model_name=f"mono-{lang}-{budget // 1_000}k",
            max_tokens=budget,
            epochs=EPOCHS,
        )
        result = subprocess.run(cmd, shell=True, executable="/bin/bash",
                                capture_output=True, text=True)
        print(result.stdout[-2000:])
        ppls = parse_perplexity(result.stdout)
        for lang_key, ppl in ppls.items():
            scaling_results.append({"lang": lang_key, "max_tokens": budget, "perplexity": ppl})

scaling_df = pd.DataFrame(scaling_results)
scaling_df

## 3 — Bilingual runs (eng + dut at fixed token budget)

In [ ]:
bilingual_results = []

for lang_a, lang_b, ratios in BILINGUAL_RATIOS:
    label = f"{lang_a}{int(ratios[0]*100)}-{lang_b}{int(ratios[1]*100)}"
    output_dir = f"./bi-{label}"
    cmd = train_cmd(
        langs=[lang_a, lang_b],
        output_dir=output_dir,
        model_name=f"bi-{label}",
        max_tokens=BILINGUAL_BUDGET,
        ratios=ratios,
        epochs=EPOCHS,
    )
    result = subprocess.run(cmd, shell=True, executable="/bin/bash",
                            capture_output=True, text=True)
    print(result.stdout[-2000:])
    ppls = parse_perplexity(result.stdout)
    for lang_key, ppl in ppls.items():
        bilingual_results.append({
            "label": label,
            "lang_a": lang_a, "lang_b": lang_b,
            "ratio_a": ratios[0], "ratio_b": ratios[1],
            "eval_lang": lang_key,
            "perplexity": ppl,
        })

bilingual_df = pd.DataFrame(bilingual_results)
bilingual_df

## 4 — Scaling law plots

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

scaling_law_params = {}  # lang → (log_a, b) where ppl = exp(log_a) * tokens^b

for lang, grp in scaling_df.groupby("lang"):
    grp = grp.sort_values("max_tokens")
    log_x = np.log(grp["max_tokens"])
    log_y = np.log(grp["perplexity"])
    b, log_a = np.polyfit(log_x, log_y, 1)
    scaling_law_params[lang] = (log_a, b)

    ax.plot(grp["max_tokens"], grp["perplexity"], marker="o", label=lang)
    x_fit = np.linspace(grp["max_tokens"].min(), grp["max_tokens"].max(), 100)
    ax.plot(x_fit, np.exp(log_a) * x_fit ** b, "--", alpha=0.5)

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("Training tokens")
ax.set_ylabel("Perplexity")
ax.set_title("Monolingual scaling laws")
ax.legend()
plt.tight_layout()
plt.savefig("scaling_laws.pdf")
plt.show()

print("\nFitted scaling laws (ppl = a · tokens^b):")
for lang, (log_a, b) in scaling_law_params.items():
    print(f"  {lang}: a={np.exp(log_a):.2f}, b={b:.3f}")

## 5 — Token Efficiency

**MLTE** (Monolingual Token Equivalence): the number of tokens a monolingual model would need to reach the same perplexity as the bilingual model achieves in that language. Derived by inverting the fitted scaling law.

**Token Efficiency** (TE = MLTE / t_A): how much more (or less) data-efficient the bilingual model is compared to a monolingual baseline.
- TE > 1: bilingual model achieves better perplexity than a monolingual model trained on the same tokens → positive transfer
- TE < 1: bilingual model underperforms the monolingual baseline → interference

In [ ]:
def mlte(lang: str, target_ppl: float) -> float:
    """Invert the fitted scaling law to find how many tokens a monolingual model needs
    to reach target_ppl: ppl = exp(log_a) * t^b → t = exp((log(ppl) - log_a) / b)"""
    log_a, b = scaling_law_params[lang]
    return np.exp((np.log(target_ppl) - log_a) / b)


rows = []
for _, r in bilingual_df.iterrows():
    is_lang_a = r["eval_lang"] == r["lang_a"]
    lang = r["lang_a"] if is_lang_a else r["lang_b"]
    ratio = r["ratio_a"] if is_lang_a else r["ratio_b"]
    t_lang = BILINGUAL_BUDGET * ratio          # actual tokens used for this language
    ppl_bi = r["perplexity"]
    mlte_val = mlte(lang, ppl_bi)
    rows.append({
        "label": r["label"],
        "eval_lang": r["eval_lang"],
        "lang": lang,
        "ratio": ratio,
        "t_lang": t_lang,
        "bilingual_ppl": ppl_bi,
        "mlte": mlte_val,
        "token_efficiency": mlte_val / t_lang,
    })

te_df = pd.DataFrame(rows)
te_df

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=False)

for ax, eval_lang in zip(axes, ["eng", "dut"]):
    sub = te_df[te_df["eval_lang"] == eval_lang].sort_values("ratio")
    bars = ax.bar(sub["label"], sub["token_efficiency"], color=["steelblue", "orange", "green"])
    ax.axhline(1.0, color="grey", linestyle="--", linewidth=1, label="TE = 1 (mono baseline)")
    ax.set_title(f"{eval_lang} Token Efficiency (MLTE / t_A)")
    ax.set_ylabel("Token Efficiency")
    ax.tick_params(axis="x", rotation=30)
    ax.legend()

plt.suptitle("Token Efficiency: >1 means positive transfer from bilingual training")
plt.tight_layout()
plt.savefig("token_efficiency.pdf")
plt.show()

## Save results

In [ ]:
scaling_df.to_csv("scaling_results.csv", index=False)
bilingual_df.to_csv("bilingual_results.csv", index=False)
te_df.to_csv("token_efficiency_results.csv", index=False)
print("Saved.")